# MLOps Assignment 2 - Fine-Tuning DistilBERT for Goodreads Genre Classification

**Author:** Aishwarya Mishra (G25AIT2137)  
**Platform:** Kaggle (GPU T4 x2)  

## Install dependencies

get_ipython().system('pip install -q -U transformers wandb huggingface_hub gdown')


## Load secrets from Kaggle


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
HF_TOKEN = os.environ["HF_TOKEN"]
print("Secrets loaded.")


## Imports


In [2]:
import random
import json
import gzip
import pickle
import requests
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

import wandb
from huggingface_hub import login as hf_login

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


## Config


In [3]:
MODEL_NAME = "distilbert-base-cased"
MAX_LENGTH = 512
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 3e-5
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
SAMPLE_PER_GENRE = 1000           # 800 train + 200 test per genre
WANDB_PROJECT = "mlops-assignment2"
WANDB_RUN_NAME = "distilbert-run-1"
HF_USERNAME = "g25ait2137"
HF_REPO_NAME = "distilbert-goodreads-genres"
HF_MODEL_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

device_name = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device_name)


Device: cuda


## Load Goodreads reviews by genre


In [4]:
genre_url_dict = {
    "poetry":                 "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz",
    "children":               "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_children.json.gz",
    "comics_graphic":         "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz",
    "fantasy_paranormal":     "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz",
    "history_biography":      "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz",
    "mystery_thriller_crime": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz",
    "romance":                "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz",
    "young_adult":            "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz",
}


def load_reviews(url, head=10000, sample_size=SAMPLE_PER_GENRE):
    reviews = []
    count = 0
    response = requests.get(url, stream=True)
    with gzip.open(response.raw, "rt", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            reviews.append(d["review_text"])
            count += 1
            if head is not None and count >= head:
                break
    return random.sample(reviews, min(sample_size, len(reviews)))


genre_reviews_dict = {}
for genre, url in genre_url_dict.items():
    print(f"Loading reviews for: {genre}")
    genre_reviews_dict[genre] = load_reviews(url, head=10000, sample_size=SAMPLE_PER_GENRE)

# Persist locally in case download fails on a re-run
pickle.dump(genre_reviews_dict, open("genre_reviews_dict.pickle", "wb"))


Loading reviews for: poetry
Loading reviews for: children
Loading reviews for: comics_graphic
Loading reviews for: fantasy_paranormal
Loading reviews for: history_biography
Loading reviews for: mystery_thriller_crime
Loading reviews for: romance
Loading reviews for: young_adult


## Train/test split + label maps


In [5]:
train_texts, train_labels = [], []
test_texts, test_labels = [], []

for genre, reviews in genre_reviews_dict.items():
    reviews = random.sample(reviews, min(SAMPLE_PER_GENRE, len(reviews)))
    split = int(0.8 * len(reviews))
    for r in reviews[:split]:
        train_texts.append(r)
        train_labels.append(genre)
    for r in reviews[split:]:
        test_texts.append(r)
        test_labels.append(genre)

unique_labels = sorted(set(train_labels))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"Train size: {len(train_texts)}, Test size: {len(test_texts)}")
print(f"Num labels: {len(unique_labels)} → {unique_labels}")


Train size: 6400, Test size: 1600
Num labels: 8 → ['children', 'comics_graphic', 'fantasy_paranormal', 'history_biography', 'mystery_thriller_crime', 'poetry', 'romance', 'young_adult']


## Tokenizer + encoding


In [6]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=MAX_LENGTH)

train_labels_encoded = [label2id[y] for y in train_labels]
test_labels_encoded = [label2id[y] for y in test_labels]


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Custom torch dataset


In [7]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = MyDataset(train_encodings, train_labels_encoded)
test_dataset = MyDataset(test_encodings, test_labels_encoded)


## Load pre-trained model


In [8]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
).to(device_name)


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Initialise W&B


In [9]:
wandb.login(key=os.environ["WANDB_API_KEY"])
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model": MODEL_NAME,
        "epochs": NUM_EPOCHS,
        "batch_size": TRAIN_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "dataset": "UCSD Goodreads",
        "platform": "Kaggle",
        "num_labels": len(id2label),
    },
)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: g25ait2137 (g25ait2137-prom-iit-rajasthan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Metrics + Trainer


In [10]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## Train


In [11]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.666675,2.470142,0.565625,0.567386
2,2.082349,2.273675,0.598750,0.592584
3,1.671111,2.260458,0.600625,0.597212


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=600, training_loss=2.4247993723551433, metrics={'train_runtime': 531.8182, 'train_samples_per_second': 36.103, 'train_steps_per_second': 1.128, 'total_flos': 2543646198988800.0, 'train_loss': 2.4247993723551433, 'epoch': 3.0})

## Evaluate + log final metrics + artifact


In [12]:
eval_results = trainer.evaluate()
print(eval_results)

wandb.log({
    "final/loss": eval_results["eval_loss"],
    "final/accuracy": eval_results["eval_accuracy"],
    "final/f1": eval_results["eval_f1"],
})

preds = trainer.predict(test_dataset).predictions.argmax(-1)
labels = [item["labels"].item() for item in test_dataset]

report_dict = classification_report(
    labels, preds,
    target_names=[id2label[i] for i in range(len(id2label))],
    output_dict=True,
)

with open("eval_report.json", "w") as f:
    json.dump(report_dict, f, indent=2)

# Also save a human-readable text version
report_text = classification_report(
    labels, preds,
    target_names=[id2label[i] for i in range(len(id2label))],
)
with open("eval_report.txt", "w") as f:
    f.write(report_text)
print(report_text)

artifact = wandb.Artifact("eval-report", type="evaluation")
artifact.add_file("eval_report.json")
artifact.add_file("eval_report.txt")
wandb.log_artifact(artifact)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 2.26045823097229, 'eval_accuracy': 0.600625, 'eval_f1': 0.5972115325998901, 'eval_runtime': 14.2608, 'eval_samples_per_second': 112.196, 'eval_steps_per_second': 1.753, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


                        precision    recall  f1-score   support

              children       0.65      0.67      0.66       200
        comics_graphic       0.81      0.79      0.80       200
    fantasy_paranormal       0.47      0.47      0.47       200
     history_biography       0.61      0.58      0.60       200
mystery_thriller_crime       0.51      0.61      0.56       200
                poetry       0.74      0.82      0.78       200
               romance       0.59      0.57      0.58       200
           young_adult       0.38      0.31      0.34       200

              accuracy                           0.60      1600
             macro avg       0.60      0.60      0.60      1600
          weighted avg       0.60      0.60      0.60      1600



<Artifact eval-report>

## Push model + tokenizer to Hugging Face Hub


In [13]:
hf_login(token=HF_TOKEN)

model.push_to_hub(HF_MODEL_ID)
tokenizer.push_to_hub(HF_MODEL_ID)

hf_url = f"https://huggingface.co/{HF_MODEL_ID}"
wandb.run.summary["huggingface_model"] = hf_url
print("Model pushed to:", hf_url)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Model pushed to: https://huggingface.co/g25ait2137/distilbert-goodreads-genres


## Finish W&B


In [14]:
wandb.finish()


eval/accuracy,▁███
eval/f1,▁▇██
eval/loss,█▁▁▁
eval/runtime,▁▆██
eval/samples_per_second,█▃▁▁
eval/steps_per_second,█▃▁▁
final/accuracy,▁
final/f1,▁
final/loss,▁
test/accuracy,▁
+10,...
